# Phase 3 â€” Reproducible Finnish CoNLL-U preprocessing

This notebook turns the six commit-pinned UD Finnish-TDT and UD Finnish-FTB files into a validated sentence-level JSONL dataset. It demonstrates and audits the reusable implementation in `app/services/data_preprocessing.py`; it does not duplicate the parser, generate learner errors, create project splits, or train a model.

## 1. Objective and raw inputs

The inputs are the three upstream splits from each treebank at the commits approved in Phase 2. Those split names remain provenance only. Phase 3 does **not** make the later project train/validation/test split.

In [1]:
from pathlib import Path
import json
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'app').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from app.services.data_preprocessing import default_source_specs, run_default_preprocessing

source_specs = default_source_specs(PROJECT_ROOT)
pd.DataFrame([
    {
        'dataset': spec.dataset,
        'commit': spec.commit,
        'split': spec.split,
        'file': spec.path.name,
        'size_bytes': spec.path.stat().st_size,
    }
    for spec in source_specs
])

,dataset,commit,split,file,size_bytes
0,UD_Finnish-TDT,bfaae13719f249573d940edda6a0d7aa8eec620f,train,fi_tdt-ud-train.conllu,13575645
1,UD_Finnish-TDT,bfaae13719f249573d940edda6a0d7aa8eec620f,dev,fi_tdt-ud-dev.conllu,1526666
2,UD_Finnish-TDT,bfaae13719f249573d940edda6a0d7aa8eec620f,test,fi_tdt-ud-test.conllu,1753468
3,UD_Finnish-FTB,2dd197c1f7c4b9b6d69be1e3f826162a4443af96,train,fi_ftb-ud-train.conllu,10725798
4,UD_Finnish-FTB,2dd197c1f7c4b9b6d69be1e3f826162a4443af96,dev,fi_ftb-ud-dev.conllu,1325526
5,UD_Finnish-FTB,2dd197c1f7c4b9b6d69be1e3f826162a4443af96,test,fi_ftb-ud-test.conllu,1370731


## 2. Linguistic-safety decisions

Finnish surface forms carry grammatical information, so preprocessing preserves the `# text` value and token `FORM` exactly. It does not globally lowercase, remove punctuation, stem, strip Finnish characters, or normalize inflection. `LEMMA` and `XPOS` use `null` when the source has `_`; `FEATS` and `MISC` become mappings, with `{}` representing a legitimate missing annotation.

NFC normalization is applied **only** while hashing `leakage_group_id`. The stored sentence is unchanged. This retains the morphology and dependencies needed later for case, conjugation, noun inflection, word order, agreement, spelling, and plural transformations.

## 3. Processed schema

Each JSONL object contains schema/provenance fields, the unchanged original text, counts, integer-ID word tokens, multiword-token ranges, and decimal-ID empty nodes. `empty_nodes` is an explicit preservation field complementing the required `empty_node_count`; this follows the Phase 2 hand-off requirement to keep alignment-sensitive rows. Integer tokens retain `id`, `form`, `lemma`, `upos`, `xpos`, parsed `feats`, `head`, `deprel`, and parsed `misc`.

## 4. Representative raw structure

This is one untouched source block before preprocessing. The tabular rows have the expected ten CoNLL-U columns.

In [2]:
raw_lines = source_specs[0].path.read_text(encoding='utf-8').splitlines()
first_boundary = raw_lines.index('')
print('\n'.join(raw_lines[:first_boundary]))

# sent_id = b101.1
# text = Kävelyreitti III
1	Kävelyreitti	kävely#reitti	NOUN	N	Case=Nom|Number=Sing	0	root	0:root	_
2	III	III	ADJ	Num	NumType=Ord	1	amod	1:amod	_


## 5. Execute the reusable pipeline

The service verifies both raw Git commits, strictly parses the six files, validates IDs/morphology/dependencies, writes deterministic UTF-8 JSONL, and builds its manifest from the actual records.

In [3]:
manifest = run_default_preprocessing(PROJECT_ROOT)
manifest['output']

{'path': 'data/processed/finnish_sentences_v1.jsonl',
 'sha256': '1d2f14e1769e26adae58df808d28fc90231fbffedfa2db46528579e1a18b97f5',
 'sentences': 33859,
 'word_tokens': 361818,
 'multiword_tokens': 555,
 'empty_nodes': 261}

## 6. Representative processed structure

The same first sentence is now a self-contained record. The compact view below shows sentence traceability and its token annotations without hiding Finnish capitalization or forms.

In [4]:
output_path = PROJECT_ROOT / manifest['output']['path']
with output_path.open(encoding='utf-8') as handle:
    first_record = json.loads(next(handle))

{key: first_record[key] for key in (
    'source_id', 'leakage_group_id', 'original_text', 'word_count',
    'empty_node_count', 'multiword_tokens', 'tokens'
)}

{'source_id': 'UD_Finnish-TDT:fi_tdt-ud-train.conllu:b101.1',
 'leakage_group_id': 'sha256:74dd0c011b67cb6988c24c6f899f211a44c60275414650f14bbc60322bec7ab6',
 'original_text': 'Kävelyreitti III',
 'word_count': 2,
 'empty_node_count': 0,
 'multiword_tokens': [],
 'tokens': [{'id': 1,
   'form': 'Kävelyreitti',
   'lemma': 'kävely#reitti',
   'upos': 'NOUN',
   'xpos': 'N',
   'feats': {'Case': 'Nom', 'Number': 'Sing'},
   'head': 0,
   'deprel': 'root',
   'misc': {}},
  {'id': 2,
   'form': 'III',
   'lemma': 'III',
   'upos': 'ADJ',
   'xpos': 'Num',
   'feats': {'NumType': 'Ord'},
   'head': 1,
   'deprel': 'amod',
   'misc': {}}]}

## 7. Source-versus-output reconciliation

All counts below come from this run. The expected Phase 2 totals are verification assertions, not parser constants.

In [5]:
file_counts = pd.DataFrame(manifest['input_files'])
display(file_counts[['source_dataset', 'source_split', 'sentences', 'word_tokens', 'multiword_tokens', 'empty_nodes']])

input_total = manifest['input_counts']['total']
output_total = {key: manifest['output'][key] for key in ('sentences', 'word_tokens', 'multiword_tokens', 'empty_nodes')}
reconciliation = pd.DataFrame({'raw_input': input_total, 'processed_output': output_total})
reconciliation['difference'] = reconciliation['processed_output'] - reconciliation['raw_input']
assert manifest['output']['sentences'] == 33859
assert manifest['output']['word_tokens'] == 361818
assert (reconciliation['difference'] == 0).all()
reconciliation

,source_dataset,source_split,sentences,word_tokens,multiword_tokens,empty_nodes
0,UD_Finnish-TDT,train,12217,162815,199,210
1,UD_Finnish-TDT,dev,1364,18308,18,22
2,UD_Finnish-TDT,test,1555,21070,27,29
3,UD_Finnish-FTB,train,14981,127613,254,0
4,UD_Finnish-FTB,dev,1875,15726,32,0
5,UD_Finnish-FTB,test,1867,16286,25,0


,raw_input,processed_output,difference
sentences,33859,33859,0
word_tokens,361818,361818,0
multiword_tokens,555,555,0
empty_nodes,261,261,0


## 8. Missing-value statistics

Underscores in legitimate optional annotations are retained as `null` or `{}` rather than causing token or sentence removal.

In [6]:
pd.DataFrame(manifest['missing_value_statistics']).T

,missing_count,missing_percent
lemma,32.0,0.008844
xpos,7.0,0.001935
feats,101410.0,28.027904
misc,319805.0,88.388361


## 9. Duplicate analysis and leakage protection

Duplicate source records are deliberately retained for provenance. Equal NFC-normalized sentence texts receive the same deterministic SHA-256 `leakage_group_id`, including duplicates that cross the two treebanks. Phase 4 must copy this ID to every derived variant; the later split must group on it.

In [7]:
pd.Series(manifest['duplicate_statistics'], name='value').to_frame()

,value
unique_text_groups,33386
duplicate_text_groups,380
records_in_duplicate_text_groups,853
duplicate_records_beyond_first,473
cross_dataset_duplicate_text_groups,339
largest_duplicate_text_group,21


## 10. Multiword-token and empty-node handling

Range IDs are not counted as syntactic words and are stored in `multiword_tokens`. Decimal IDs are not mixed into `tokens`; they are stored in `empty_nodes` and counted separately.

In [8]:
mwt_example = None
empty_example = None
with output_path.open(encoding='utf-8') as handle:
    for line in handle:
        record = json.loads(line)
        if mwt_example is None and record['multiword_tokens']:
            mwt_example = {
                'source_id': record['source_id'],
                'text': record['original_text'],
                'word_count': record['word_count'],
                'multiword_tokens': record['multiword_tokens'],
            }
        if empty_example is None and record['empty_nodes']:
            empty_example = {
                'source_id': record['source_id'],
                'text': record['original_text'],
                'empty_node_count': record['empty_node_count'],
                'empty_nodes': record['empty_nodes'],
            }
        if mwt_example is not None and empty_example is not None:
            break

display(mwt_example)
display(empty_example)

{'source_id': 'UD_Finnish-TDT:fi_tdt-ud-train.conllu:b108.18',
 'text': 'Olenko minä sanomassa, etten tahdo majoittaa äitiäni, joka jaksaa tukea ja kannustaa puolisoaan; toivoo hänen kanssaan, että lääkkeet hidastavat sairauden etenemistä niin paljon, että kuolema ehtisi korjata vanhan ihmisen ennen kuin sairaus pyyhkii ajatukset näkymättömiin.',
 'word_count': 42,
 'multiword_tokens': [{'id': '5-6',
   'start': 5,
   'end': 6,
   'form': 'etten',
   'misc': {}}]}

{'source_id': 'UD_Finnish-TDT:fi_tdt-ud-train.conllu:b309.8',
 'text': 'Jos haluamme esim. alusvaatteilta pelkästään miellyttävää tunnetta iholla tai yksinkertaisesti vain alusvaatteet, jäävät ulkoiset seikat sivuun.',
 'empty_node_count': 1,
 'empty_nodes': [{'id': '9.1',
   'form': 'haluamme',
   'lemma': 'haluta',
   'upos': 'VERB',
   'xpos': None,
   'feats': {'Mood': 'Ind',
    'Number': 'Plur',
    'Person': '1',
    'Tense': 'Pres',
    'VerbForm': 'Fin',
    'Voice': 'Act'},
   'head': None,
   'deprel': None,
   'deps': '2:conj',
   'misc': {}}]}

## 11. Quality checks

Malformed rows are never repaired silently: failures include file, line, and sentence context. A successful run means all ten-column, identifier, dependency-head, FEATS, range, provenance, leakage-ID, and reconciliation checks below passed.

In [9]:
quality = pd.Series(manifest['quality_checks'], name='passed').to_frame()
assert quality['passed'].all()
assert first_record['original_text'] == raw_lines[1].split('=', 1)[1].lstrip()
assert first_record['tokens'][0]['form'] == raw_lines[2].split('\t')[1]
quality

,passed
all_rows_have_ten_columns,True
token_ids_validated,True
dependency_heads_validated,True
dependency_trees_acyclic,True
feats_syntax_validated,True
multiword_token_ranges_validated,True
sentence_ids_present_or_deterministically_fallback,True
source_ids_unique,True
all_records_have_source_traceability,True
all_records_have_leakage_group_id,True


## 12. Final processed-data summary

The artifacts are `data/processed/finnish_sentences_v1.jsonl` and `data/processed/preprocessing_manifest_v1.json`. The JSONL contains one record per source sentence; the manifest records commits, file hashes, per-file and aggregate counts, missing values, duplicates, MWT/empty-node totals, output hash, and validation results.

In [10]:
pd.Series({
    'processed_file': manifest['output']['path'],
    'sentences': manifest['output']['sentences'],
    'word_tokens': manifest['output']['word_tokens'],
    'unique_leakage_groups': manifest['duplicate_statistics']['unique_text_groups'],
    'duplicate_text_groups': manifest['duplicate_statistics']['duplicate_text_groups'],
    'multiword_tokens': manifest['output']['multiword_tokens'],
    'empty_nodes': manifest['output']['empty_nodes'],
    'sha256': manifest['output']['sha256'],
}, name='Phase 3 result').to_frame()

,Phase 3 result
processed_file,data/processed/finnish_sentences_v1.jsonl
sentences,33859
word_tokens,361818
unique_leakage_groups,33386
duplicate_text_groups,380
multiword_tokens,555
empty_nodes,261
sha256,1d2f14e1769e26adae58df808d28fc90231fbffedfa2db...


## 13. Hand-off requirements for Phase 4

Phase 4 may read the processed JSONL but must not mutate it. Every synthetic variant must retain `source_id`, copy `leakage_group_id`, identify its controlled generation rule, and preserve a correct reference sentence. Synthetic generation must remain linguistically motivated and validated. No final train/validation/test split should separate records that share a leakage group.